# 🧠 mdlARC 复现 · 一键版（T4适配）

> 软软特制！代码（含T4补丁）+ 数据全部打包，**无需GitHub拉取、无需手动上传**，点开就能跑。
>
> 📌 步骤：运行时 → 更改运行时类型 → **T4 GPU** → 依次运行所有cell
>
> 原项目：[mvakde/mdlARC](https://github.com/mvakde/mdlARC)（1.5h训出 ARC-AGI-1 44%，$0.67）
> 补丁：SDPA回退 + fp16（T4无flash-attn也能跑）

## 0️⃣ 检查GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU!')
assert torch.cuda.is_available(), '请先选择GPU运行时: 代码执行程序→更改运行时类型→T4 GPU'
print('✅ GPU就绪')

## 1️⃣ 安装依赖

In [ ]:
!pip install -q numba matplotlib
print('依赖安装完成 ✅')

## 2️⃣ 自动下载离线包并解压

In [ ]:
import os, tarfile, urllib.request

URL = 'https://github.com/patient-Zero-0/mdlARC-colab/raw/main/mdlARC_offline.tar.gz'
PKG = 'mdlARC_offline.tar.gz'

if not os.path.exists(PKG):
    print('⬇️ 下载离线包（618KB）...')
    urllib.request.urlretrieve(URL, PKG)
    print('下载完成 ✅')

with tarfile.open(PKG) as tf:
    tf.extractall('mdlARC')
print('解压完成 ✅')
print('内容:', os.listdir('mdlARC'))

import json
c = json.load(open('mdlARC/assets/challenges.json'))
print(f'数据集就绪：{len(c)} 题训练挑战 ✅')

## 3️⃣ 验证补丁（GPU前向）

In [ ]:
%cd mdlARC
import sys, torch
sys.path.insert(0, 'src')
from common import VOCAB_SIZE
from tinytransformer import TinyTransformer, TinyTransformerConfig

cfg = TinyTransformerConfig(
    d_model=128, n_heads=4, d_ff=512, n_layers=2,
    num_examples=1307, num_dihedrals=8,
)
model = TinyTransformer(cfg).cuda()
print('flash_attn可用:', model.blocks[0].attention._has_flash_attn_varlen, '(False=SDPA回退生效)')
T = 30
x = torch.randint(0, VOCAB_SIZE, (T,)).cuda()
example_ids = torch.tensor([0, 1]).cuda()
dihedral_ids = torch.tensor([0, 0]).cuda()
cu_seqlens = torch.tensor([0, 10, 30], dtype=torch.int32).cuda()
sep_indices = torch.tensor([6, 18]).cuda()
pos = torch.zeros(T, 3, dtype=torch.long).cuda()
out = model(x, example_ids, dihedral_ids, cu_seqlens=cu_seqlens, max_seqlen=20, positions_3d=pos, sep_indices=sep_indices)
out['output_loss'].backward()
print(f'GPU前向+反向OK, loss={out["output_loss"].item():.3f} ✅')

## 4️⃣ 训练 + 评测

预设：`low`(90轮) / `medium`(240轮) / `high`(650轮)。
T4预估：low≈1-2h，medium≈3-5h，high≈8-12h。**建议先跑low**！

In [ ]:
PRESET = 'low'  # ← 改这里：low / medium / high

import subprocess, sys
print(f'🚀 开始训练 ({PRESET})...')
r = subprocess.run([sys.executable, 'run_script.py', PRESET], capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('❌ 失败：', r.stderr[-2000:])
else:
    print('\n✅ 训练+评测完成！')

## 5️⃣ 查看ARC得分

In [ ]:
import sys, os
sys.path.insert(0, 'src')
import utils
from pathlib import Path

submission = Path('runs') / [p for p in os.listdir('runs') if 'submission' in p][0]
score = utils.score_arc_submission(Path('assets/solutions.json'), submission)
print(f'🏆 ARC-1 得分: {score}%')